សៀវភៅកំណត់ត្រាដូចតទៅត្រូវបានបង្កើតឡើងដោយស្វ័យប្រវត្តិដោយ GitHub Copilot Chat និងមានគោលបំណងសម្រាប់ការតំបន់ដំបូងតែប៉ុណ្ណោះ


# ការណែនាំស្តីពីវិស្វកម្មបញ្ចូលបែប
វិស្វកម្មបញ្ចូលបែបគឺជាដំណើរការរចនា និងបង្កើតបញ្ចូលបែបសម្រាប់ការងារដែលជាសម្ភាសន៍ភាសាតាមធម្មជាតិ។ វាអាចចូលរួមក្នុងការជ្រើសរើសបញ្ចូលបែបត្រឹមត្រូវ ការកំណត់ប៉ារ៉ាម៉ែត្ររបស់ពួកវា ហើយការវាយតម្លៃការសម្រួលរបស់ពួកវា។ វិស្វកម្មបញ្ចូលបែបមានសារៈសំខាន់សម្រាប់ការសម្រេចបាននូវភាពត្រឹមត្រូវខ្ពស់ និងប្រសិទ្ធភាពនៅក្នុងម៉ូដែល NLP។ នៅក្នុងផ្នែកនេះ យើងនឹងសិក្សាអំពីមូលដ្ឋាននៃវិស្វកម្មបញ្ចូលបែបដោយប្រើម៉ូដែល OpenAI សម្រាប់ការស្រាវជ្រាវ។


### ការប្រឡង 1៖ ការបំបែកពាក្យ
ស្វែងយល់ពីការបំបែកពាក្យដោយប្រើ tiktoken ដែលជាឧបករណ៍បំបែកពាក្យលឿនបើកចំហពី OpenAI
ឈានទៅកាន់ [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) សម្រាប់ឧទាហរណ៍បន្ថែមទៀត។


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### លំហាត់ ២៖ ធ្វើតេស្តការតំឡើងកូនសោ API របស់ OpenAI

រត់កូដខាងក្រោមដើម្បីផ្ទៀងផ្ទាត់ថាចំណុចចុង OpenAI របស់អ្នកត្រូវបានដំឡើងត្រឹមត្រូវ។ កូដនេះព្យាយាមបញ្ចូលការបញ្ជាលើសាមញ្ញមួយ ហើយផ្ទៀងផ្ទាត់ការបញ្ចប់។ បញ្ចូល `oh say can you see` គួរតែបញ្ចប់ជាអត្ថបទដែលស្រដៀងៗគ្នា `by the dawn's early light..`


In [ ]:
# Uses the OpenAI client against the Azure OpenAI (Microsoft Foundry) v1 endpoint
# with the Responses API. See https://aka.ms/openai/start

import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']

def get_completion(prompt):
    response = client.responses.create(
        model=deployment,
        input=prompt,
        max_output_tokens=1024,
        store=False,
    )
    return response.output_text

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt)
print(response)


### ការហាត់ប្រាណទី 3: ការបង្កើតសំណុំបែបបទ
ស្វែងយល់អំពីអ្វីដែលកើតឡើងនៅពេលអ្នកសួរឲ្យ LLM ត្រឡប់ការបញ្ចប់សម្រាប់បញ្ហាសំណួរអំពីប្រធានបទដែលប្រហែលជា មិនមាន ឬអំពីប្រធានបទដែលវាអាចមិនដឹង អំពីព្រោះវាបាននៅក្រៅសំណុំនៃទិន្នន័យដែលវាបានហ្វឹកហ្វឺនមុននោះ (ថ្មីជាងនេះ)។ មើលថាការឆ្លើយតបប្ដូរយ៉ាងដូចម្តេច ប្រសិនបើអ្នកព្យាយាមប្រើបញ្ហាសំណួរផ្សេង ឬម៉ូដែលផ្សេងទៀត។


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt)
print(response)

### វិភាគ 4: បញ្ជាទ្រង់មូល 
ប្រើអថេរ "text" ដើម្បីកំណត់មាតិកាលើកដំបូង 
និងអថេរ "prompt" ដើម្បីផ្តល់ការណែនាំដែលទាក់ទងនឹងមាតិការលើកដំបូងនោះ។

នៅទីនេះយើងស្នើឲ្យម៉ូដែលសង្ខេបអត្ថបទសម្រាប់សិស្សថ្នាក់ទីពីរ


In [ ]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt)
print(response)

### លំហាត់ 5: សំណើស្មុគស្មាញ  
សូមព្យាយាមសំណើដែលមានសារ​ពីប្រព័ន្ធ ប្រើប្រាស់ និងជំនួយការបានសារ  
ប្រព័ន្ធកំណត់បរិបទជំនួយការ  
សារ​ប្រើប្រាស់ និងជំនួយការផ្តល់បរិបទសន្ទនា​ច្រើនជំហាន  

សូមកត់សម្គាល់ថាបុគ្គលិកលក្ខណៈជំនួយការត្រូវបានកំណត់ជា "អាស្រ័យបំភាន់" ក្នុងបរិបទ​ប្រព័ន្ធ។  
សូមព្យាយាមប្រើបរិបទបុគ្គលិកលក្ខណៈផ្សេងទៀត។ ឬសាកល្បងសំណួរបញ្ចូល និងចម្លើយបន្តផ្សេងទៀត  


In [ ]:
response = client.responses.create(
    model=deployment,
    input=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ],
    store=False,
)
print(response.output_text)


### អនុវត្ត៖ ស្វែងយល់ពីការស៊ើបអង្កេតឱ្យបានច្បាស់របស់អ្នក
ឧទាហរណ៍ខាងលើផ្តល់ឱ្យអ្នកនូវលំនាំដែលអ្នកអាចប្រើប្រាស់ដើម្បីបង្កើតសំណើថ្មីៗ (សាមញ្ញ, រំខាន់, សេចក្តីណែនាំ ល។) - សាកល្បងបង្កើតអនុវត្តផ្សេងទៀតដើម្បីស្រាវជ្រាវខ្លះពីគំនិតផ្សេងទៀតដែលយើងបានពិភាក្សា ដូចជា ឧទាហរណ៍, តំណាង និងផ្សេងទៀត។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
